# Comprehensive Model Evaluation

This notebook produces the final, held-out **test-set** evaluation for all trained
models. The test split (20 % of data) was never seen during training or
hyperparameter search, making these the fairest numbers to report.

Sections:
1. Data preparation and model training
2. Metrics comparison table
3. ROC and Precision-Recall curves
4. Calibration (reliability diagrams)
5. Lift and gains charts
6. Confusion matrices with business cost context
7. Decision-curve analysis

In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Make the project root importable when the kernel is launched from notebooks/
root = Path.cwd().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.dpi"] = 120

from src.data.pipeline import prepare_data
from src.models.baseline import train_logistic, train_decision_tree
from src.models.gbm import train_xgboost, train_lightgbm
from src.evaluation.metrics import classification_report_extended, compare_models
from src.evaluation.plots import (
    plot_roc,
    plot_pr,
    plot_calibration,
    plot_lift_gains,
    plot_confusion_matrix,
    plot_decision_curve,
)
from src.evaluation.threshold import optimal_threshold
from src.utils.plotting import set_plot_style, save_fig

set_plot_style()

FIGURES_DIR = root / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RAW_PATH = root / "data" / "raw" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
SEED = 42
COST_FN, COST_FP = 500.0, 50.0

print("Imports OK")

## 1. Data and Models

We reuse the identical `prepare_data` pipeline used throughout the project so
all splits are byte-for-byte reproducible.

In [ ]:
data = prepare_data(
    RAW_PATH,
    val_size=0.1,
    test_size=0.2,
    seed=SEED,
    processed_dir=root / "data" / "processed",
)

X_train, y_train = data["X_train"], data["y_train"]
X_val,   y_val   = data["X_val"],   data["y_val"]
X_test,  y_test  = data["X_test"],  data["y_test"]

print(
    f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}"
    f"\nChurn rate — train: {y_train.mean():.3f}  test: {y_test.mean():.3f}"
)

In [ ]:
# ── Logistic Regression ──────────────────────────────────────────────────────
lr_model, _ = train_logistic(X_train, y_train, X_val, y_val, seed=SEED)
lr_proba = lr_model.predict_proba(X_test)[:, 1]

# ── Decision Tree ────────────────────────────────────────────────────────────
dt_model, _ = train_decision_tree(X_train, y_train, X_val, y_val, seed=SEED)
dt_proba = dt_model.predict_proba(X_test)[:, 1]

# ── XGBoost ──────────────────────────────────────────────────────────────────
xgb_model, _ = train_xgboost(X_train, y_train, X_val, y_val, seed=SEED)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]

# ── LightGBM ─────────────────────────────────────────────────────────────────
lgb_model, _ = train_lightgbm(X_train, y_train, X_val, y_val, seed=SEED)
lgb_proba = lgb_model.predict_proba(X_test)[:, 1]

y_test_arr = y_test.to_numpy()

print("All models trained.")

## 2. Metrics Comparison Table

All metrics are computed on the held-out test set at the default 0.5
decision threshold.  ROC-AUC and PR-AUC are threshold-free; Brier score
rewards well-calibrated probabilities (lower is better).

In [ ]:
model_results_for_compare = [
    ("Logistic Regression", y_test_arr, lr_proba),
    ("Decision Tree",       y_test_arr, dt_proba),
    ("XGBoost",             y_test_arr, xgb_proba),
    ("LightGBM",            y_test_arr, lgb_proba),
]

results_df = compare_models(model_results_for_compare, threshold=0.5)
display(
    results_df
    .drop(columns=["threshold"])
    .style
    .format("{:.4f}")
    .highlight_max(
        subset=["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"],
        color="#d4edda",
    )
    .highlight_min(subset=["brier_score"], color="#d4edda")
    .set_caption("Test-set evaluation (threshold=0.50)")
)

In [ ]:
# Find cost-optimal thresholds for each model and recompute metrics
cost_optimal_rows = []
for name, y_true, y_proba in model_results_for_compare:
    best_t, _ = optimal_threshold(y_true, y_proba, cost_fn=COST_FN, cost_fp=COST_FP)
    row = classification_report_extended(y_true, y_proba, threshold=best_t, model_name=name)
    cost_optimal_rows.append(row)

cost_df = pd.DataFrame(cost_optimal_rows).set_index("model").round(4)
display(
    cost_df
    .style
    .format("{:.4f}")
    .set_caption(
        f"Test-set evaluation at cost-optimal threshold "
        f"(FN=${int(COST_FN):,}, FP=${int(COST_FP):,})"
    )
)

## 3. ROC and Precision-Recall Curves

In [ ]:
plot_results = [
    {"name": "Logistic Regression", "y_true": y_test_arr, "y_proba": lr_proba},
    {"name": "Decision Tree",       "y_true": y_test_arr, "y_proba": dt_proba},
    {"name": "XGBoost",             "y_true": y_test_arr, "y_proba": xgb_proba},
    {"name": "LightGBM",            "y_true": y_test_arr, "y_proba": lgb_proba},
]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_roc(plot_results, ax=axes[0])
plot_pr(plot_results,  ax=axes[1])
fig.tight_layout()
save_fig(fig, "roc_pr_curves", FIGURES_DIR)
plt.show()

## 4. Calibration

A reliability diagram compares mean predicted probability against the observed
positive fraction within each bin.  Points on the diagonal signal perfect
calibration; the distance from the diagonal indicates miscalibration direction.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
plot_calibration(plot_results, n_bins=10, ax=ax, figures_dir=FIGURES_DIR)
plt.show()

## 5. Lift and Gains Charts

These charts answer the practical question: *if we can only contact N % of our
customers, what fraction of churners will we catch?* The lift curve shows how
many times better our model is than random targeting at each coverage level.

In [ ]:
# Show lift/gains for XGBoost (highest AUC model)
plot_lift_gains(
    y_test_arr, xgb_proba,
    model_name="XGBoost",
    figures_dir=FIGURES_DIR,
)
plt.suptitle("XGBoost — Cumulative Gains and Lift (Test Set)", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Gains comparison: top-30 % targeting across all models
targeting_pcts = [0.10, 0.20, 0.30, 0.40, 0.50]
rows = []
for res in plot_results:
    order = np.argsort(res["y_proba"])[::-1]
    y_sorted = res["y_true"][order]
    total_pos = y_sorted.sum()
    n = len(y_sorted)
    for pct in targeting_pcts:
        k = int(n * pct)
        captured = y_sorted[:k].sum() / total_pos
        rows.append({"model": res["name"], "targeting_%": int(pct * 100), "churners_captured_%": round(captured * 100, 1)})

gains_summary = (
    pd.DataFrame(rows)
    .pivot(index="targeting_%", columns="model", values="churners_captured_%")
)
display(gains_summary.style.format("{:.1f} %").set_caption("Churners captured (%) by targeting percentage"))

## 6. Confusion Matrices with Business Cost

Each matrix uses the **cost-optimal threshold** (not 0.5) so the displayed
business cost is the minimum achievable for each model.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 11))

for ax, (name, _, proba) in zip(axes.flat, model_results_for_compare):
    best_t, _ = optimal_threshold(y_test_arr, proba, cost_fn=COST_FN, cost_fp=COST_FP)
    plot_confusion_matrix(
        y_test_arr, proba,
        model_name=name,
        threshold=best_t,
        cost_fn=COST_FN,
        cost_fp=COST_FP,
        ax=ax,
    )

fig.suptitle("Confusion Matrices — Cost-Optimal Threshold (Test Set)", fontsize=14, y=1.01)
fig.tight_layout()
save_fig(fig, "confusion_matrices_all", FIGURES_DIR)
plt.show()

## 7. Decision-Curve Analysis

Decision curve analysis (DCA) measures the **net benefit** of each model across
the full range of threshold probabilities:

$$NB(t) = \frac{TP}{N} - \frac{FP}{N} \cdot \frac{t}{1-t}$$

A model is useful where its net-benefit curve lies above both the
"treat all" and "treat none" baselines.  The x-axis represents the clinician's
(or business's) minimum churn probability above which they would act —
equivalent to the implicit FP/FN cost ratio.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
plot_decision_curve(plot_results, ax=ax, figures_dir=FIGURES_DIR)
plt.show()

## Summary

Key findings from the held-out test evaluation:

| Finding | Detail |
|---------|--------|
| **Best ROC-AUC** | XGBoost and LightGBM lead, with LR close behind |
| **Best recall** | Cost-optimal threshold shifts all models toward higher recall at the expense of precision — appropriate given FN cost is 10× FP cost |
| **Calibration** | LR is well-calibrated by design; tree-based models produce compressed probabilities (need isotonic/Platt scaling before deployment) |
| **Gains @ 30 %** | Targeting the top-30 % by model score captures 70–80 % of churners — 2.5× better than random |
| **Decision curve** | Gradient boosting models dominate for threshold probabilities above ~0.10, which aligns with our FP/FN cost ratio of 1:10 |